# 09 · 可视化模块（Viz）功能演示

演示分箱图、模型评估图(ROC/KS/Lift/校准)、评分图、相关/稳定性图与策略人群图，图片导出到 model_report。

In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hscredit

# 路径约定：从 notebooks/ 目录运行，数据在 ../examples，产物输出到 model_report/
DATA = os.path.join("..", "examples", "hscredit_yyp.xlsx")
if not os.path.exists(DATA):
    DATA = os.path.join("examples", "hscredit_yyp.xlsx")
OUT = "model_report"
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA)
df["放款时间"] = pd.to_datetime(df["放款时间"])
y = df["FPD"].astype(int)
NUM_FEATURES = ["珊瑚92", "青云24", "衡枢鉴真分老客版", "占信V3", "天创小额网贷分", "近六个月非银多头机构数"]
CAT_FEATURE = "商品类别"
print("数据形状:", df.shape)
print("坏样本率: {:.4f}".format(y.mean()))
df.head()

数据形状: (970, 18)
坏样本率: 0.1402


,客户编号,放款时间,放款金额,商品类别,MOB1,CURRENT_DPD,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1985945640026276096,2026-02-03,1399,礼包,0,0,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1985972188268592896,2026-02-04,1399,礼包,0,0,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,1986034700861140992,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,1986264852923760896,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1986265696509906944,2026-01-26,1399,礼包,0,0,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0


## 1. 准备模型、评分卡与分箱表

In [2]:
from sklearn.model_selection import train_test_split
import hscredit.core.viz as V
from hscredit.core.binning import OptimalBinning
from hscredit.core.models import LightGBMRiskModel, ScoreCard, LogisticRegression

def savefig(fig, name):
    # 兼容返回 Figure / Axes / None（部分绘图函数在当前画布作图并返回 None）三种情况
    if fig is None:
        f = plt.gcf()
    elif hasattr(fig, 'savefig'):
        f = fig
    else:
        f = fig.figure
    path = f"{OUT}/09_viz_{name}.png"
    f.savefig(path, dpi=100, bbox_inches='tight')
    plt.close(f)
    return path

Xn = df[NUM_FEATURES].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(Xn, y, test_size=0.3, random_state=0, stratify=y)
m = LightGBMRiskModel(n_estimators=80).fit(Xtr, ytr)
p_tr, p_te = m.predict_proba(Xtr)[:,1], m.predict_proba(Xte)[:,1]
yte_a = yte.values
binner = OptimalBinning(method='best_iv').fit(Xtr, ytr)
bt = binner.get_bin_table('衡枢鉴真分老客版')
wtr = binner.transform(Xtr, metric='woe'); wte = binner.transform(Xte, metric='woe')
sc = ScoreCard().fit(wtr, ytr); score_te = sc.predict_score(wte)
lr = LogisticRegression().fit(wtr, ytr)
df['score'] = df['衡枢鉴真分老客版'].fillna(df['衡枢鉴真分老客版'].median())
print('准备完成')

[1]	valid_0's binary_logloss: 0.406946
[2]	valid_0's binary_logloss: 0.410507
[3]	valid_0's binary_logloss: 0.416937
[4]	valid_0's binary_logloss: 0.412811
[5]	valid_0's binary_logloss: 0.415373
[6]	valid_0's binary_logloss: 0.416595
[7]	valid_0's binary_logloss: 0.418369
[8]	valid_0's binary_logloss: 0.419069
[9]	valid_0's binary_logloss: 0.422504
[10]	valid_0's binary_logloss: 0.423557
[11]	valid_0's binary_logloss: 0.426201
[12]	valid_0's binary_logloss: 0.429132
[13]	valid_0's binary_logloss: 0.433332
[14]	valid_0's binary_logloss: 0.437366
[15]	valid_0's binary_logloss: 0.439635
[16]	valid_0's binary_logloss: 0.445192
[17]	valid_0's binary_logloss: 0.441973
[18]	valid_0's binary_logloss: 0.447634
[19]	valid_0's binary_logloss: 0.446409
[20]	valid_0's binary_logloss: 0.445939
[21]	valid_0's binary_logloss: 0.443579
[22]	valid_0's binary_logloss: 0.447583
[23]	valid_0's binary_logloss: 0.448855
[24]	valid_0's binary_logloss: 0.449209
[25]	valid_0's binary_logloss: 0.446684
[26]	vali

准备完成


## 2. 分箱图 / WOE 趋势 / 多逾期分箱图

In [3]:
fig = V.bin_plot(bt); savefig(fig, 'bin_plot')
fig = V.variable_woe_trend_plot(bt); savefig(fig, 'woe_trend')
fig = V.bin_overdues_plot(df, feature='衡枢鉴真分老客版', overdue=['MOB1','MOB1','MOB1'], dpds=[7,3,0]); savefig(fig, 'bin_overdues')
print('已保存分箱相关图')

已保存分箱相关图


## 3. 模型评估图：ROC / KS / Lift / Gain / 混淆矩阵 / 校准曲线

In [4]:
for fn, name in [(lambda: V.roc_plot(yte_a, p_te),'roc'), (lambda: V.ks_plot(p_te, yte_a),'ks'),
                 (lambda: V.lift_plot(yte_a, p_te),'lift'), (lambda: V.gain_plot(yte_a, p_te),'gain'),
                 (lambda: V.confusion_matrix_plot(yte_a, (p_te>0.5).astype(int)),'cm'),
                 (lambda: V.calibration_plot(yte_a, p_te),'calibration')]:
    savefig(fn(), name)
print('已保存模型评估图')

已保存模型评估图


## 4. 逻辑回归系数误差图 plot_weights

In [5]:
savefig(V.plot_weights(lr.summary()), 'weights'); print('已保存系数图')

已保存系数图


## 5. 评分相关图：评分分布 / KS / Lift / 通过率-坏率曲线

In [6]:
savefig(V.score_dist_plot(pd.DataFrame({'score':score_te,'target':yte_a}),'score','target'), 'score_dist')
savefig(V.score_ks_plot(y_true=yte_a, y_prob=p_te), 'score_ks')
savefig(V.score_distribution_comparison_plot({'训练':p_tr,'测试':p_te}), 'score_dist_cmp')
savefig(V.score_approval_badrate_curve(yte_a, score_te), 'approval_badrate')
print('已保存评分相关图')

已保存评分相关图


## 6. 特征/相关性/稳定性图：相关热图 / IV / 缺失坏率 / 时间趋势

In [7]:
savefig(V.corr_plot(Xn), 'corr')
savefig(V.variable_iv_plot(df, NUM_FEATURES, 'FPD'), 'variable_iv')
savefig(V.feature_trend_by_time(df, '衡枢鉴真分老客版', '放款时间'), 'feature_trend')
savefig(V.psi_plot(p_tr, p_te), 'psi')
print('已保存特征/稳定性图')

已保存特征/稳定性图


## 7. 策略与人群图：审批通过率趋势 / 坏率趋势 / 人群漂移

In [8]:
savefig(V.approval_rate_trend_plot(df, '放款时间', score_col='score', threshold=600), 'approval_trend')
savefig(V.bad_rate_trend_plot(df, '放款时间', 'FPD'), 'badrate_trend')
savefig(V.population_drift_monitor([df.iloc[:500], df.iloc[500:]], ['p1','p2'], NUM_FEATURES[:3]), 'population_drift')
print('所有图已导出到 model_report/')

所有图已导出到 model_report/
